# Adaptive Threshold — Phase 1: Local FPR-Targeted Blend (v2, WINDOW=60)

**Goal**: cut drift FPR (currently ~0.12 with static `val_p99`, ~high FP under mean+k·std)
without destroying in-distribution performance (ID gate: PR-AUC ≥ 0.90, F1 ≥ 0.88).

**What changes vs previous v2 blended notebook**:
1. Keep the corrected `K_ADAPTIVE = (val_p99 - mu_train) / sigma_train` mean+k·std blend as a baseline.
2. Add a **local FPR-targeted** threshold: blend the local empirical `(1 - target_fpr)` percentile
   of recent normal-classified MSEs with the global `val_p99`, using the same shrinkage weight.
3. Calibrate `PRIOR_STRENGTH` on `cc1_val` only (leak-free) to hit ~1% FPR.
4. Report Precision / Recall / F1 / **FPR** for static, mean+k·std, and FPR-targeted methods.

Still fully leak-free — no test/drift labels used for any hyperparameter.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, joblib, os, copy, time
import matplotlib.pyplot as plt
from pathlib import Path
from collections import deque
from scipy import stats
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score, precision_recall_curve,
)

def resolve_base():
    here = Path.cwd().resolve()
    candidates = [
        here if here.name == 'module3' else None,
        here.parent if here.name == 'module3_pipeline_v2' else None,
        Path(r'c:\Users\DELL\Documents\Claude\Projects\FYP\module3'),
        Path(r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'),
    ]
    for c in candidates:
        if c is not None and (c / 'models_v2').exists():
            return str(c)
    raise FileNotFoundError('Could not locate module3 BASE (models_v2 missing).')

BASE = resolve_base()
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
WIN_DIR_V1 = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models_v2')
MODEL_DIR_V1 = os.path.join(BASE, 'models')
print('BASE =', BASE)

WINDOW_SIZE = 60
BUFFER_SIZE = 500
TARGET_FPR = 0.01
MIN_LOCAL_FOR_PERCENTILE = 30

SPLIT_FILES = {
    'cc1_val': 'cc1_val.csv',
    'cc1_test': 'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}
ALL_EVAL = ['cc1_test', 'drift_cc2']
print('Phase 1 constants ready.')


## Step 1 — Load model + derive corrected k


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden2), nn.ReLU(),
            nn.Linear(hidden2, hidden1), nn.ReLU(),
            nn.Linear(hidden1, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)

    def decode(self, z):
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval()
        mu, _ = self.encode(x)
        return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
prior_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
STATIC_VAL_P99 = float(prior_eval['thresholds']['val_p99'])
GLOBAL_MEAN, GLOBAL_STD = float(meta['mu_train']), float(meta['sigma_train'])
K_ADAPTIVE = (STATIC_VAL_P99 - GLOBAL_MEAN) / GLOBAL_STD
CLIP = meta['clip']

model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
model.eval()
print(f'mu_train={GLOBAL_MEAN:.5f}  sigma_train={GLOBAL_STD:.5f}  val_p99={STATIC_VAL_P99:.5f}')
print(f'Corrected K_ADAPTIVE={K_ADAPTIVE:.4f}')


## Step 2 — Load windows + recover per-window cmdb_id


In [ ]:
def window_meta(df, window_size, stride=1):
    cmdb_ids, end_ts, ys = [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i + window_size].any():
                continue
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i + window_size - 1])
            ys.append(int(labels[i:i + window_size].any()))
    return np.array(cmdb_ids), np.array(end_ts), np.array(ys, dtype=np.int64)

assert os.path.isdir(WIN_DIR), f'Missing {WIN_DIR} — place windows_cc1_v2 under data/processed/'
assert os.path.isdir(DATA_DIR), f'Missing {DATA_DIR}'

cmdb_ids, end_ts, y, ft, mse = {}, {}, {}, {}, {}
for name, fname in SPLIT_FILES.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    df['is_gap'] = df['is_gap'].astype(bool)
    cmdb_ids[name], end_ts[name], y_replayed = window_meta(df, WINDOW_SIZE)
    y[name] = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    ft[name] = np.load(os.path.join(WIN_DIR, f'ft_{name}.npy'), allow_pickle=True)
    X = np.clip(np.load(os.path.join(WIN_DIR, f'X_{name}.npy')), -CLIP, CLIP).astype(np.float32)
    assert np.array_equal(y[name], y_replayed), f'{name}: cmdb/order mismatch'
    with torch.no_grad():
        mse[name] = model.anomaly_score(torch.from_numpy(X)).numpy()
    print(f'  {name:10s}: {len(y[name]):,} windows | anom={int(y[name].sum())} | cmdb check OK')


## Step 3 — Threshold algorithms: mean+k·std blend vs local FPR-targeted blend


In [ ]:
def run_mean_k_blend(mse_arr, cmdb_arr, prior_strength, buffer_size=BUFFER_SIZE,
                     k=K_ADAPTIVE, global_mean=GLOBAL_MEAN, global_std=GLOBAL_STD):
    """Original v2 shrinkage blend: blended_mean + k * blended_std."""
    n = len(mse_arr)
    preds = np.zeros(n, dtype=np.int64)
    thresh_used = np.zeros(n, dtype=np.float64)
    buffers = {}
    for i in range(n):
        cid = cmdb_arr[i]
        buf = buffers.setdefault(cid, deque(maxlen=buffer_size))
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local == 0:
            lm, ls = global_mean, global_std
        else:
            arr = np.fromiter(buf, dtype=np.float64)
            lm, ls = arr.mean(), (arr.std() if n_local > 1 else global_std)
        t = (w * lm + (1 - w) * global_mean) + k * (w * ls + (1 - w) * global_std)
        thresh_used[i] = t
        is_anom = mse_arr[i] > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse_arr[i])
    return preds, thresh_used


def run_fpr_targeted(mse_arr, cmdb_arr, prior_strength, target_fpr=TARGET_FPR,
                     buffer_size=BUFFER_SIZE, global_threshold=STATIC_VAL_P99,
                     min_local=MIN_LOCAL_FOR_PERCENTILE):
    """Phase-1 method: shrinkage blend of local (1-target_fpr) percentile with global val_p99.

    As local history grows, the threshold tracks the empirical local FPR operating point,
    which is exactly what fails under drift when a static CC1-val threshold is reused.
    """
    n = len(mse_arr)
    preds = np.zeros(n, dtype=np.int64)
    thresh_used = np.zeros(n, dtype=np.float64)
    weight_used = np.zeros(n, dtype=np.float64)
    buffers = {}
    q = (1.0 - target_fpr) * 100.0
    for i in range(n):
        cid = cmdb_arr[i]
        buf = buffers.setdefault(cid, deque(maxlen=buffer_size))
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local < min_local:
            local_t = global_threshold
        else:
            local_t = float(np.percentile(np.fromiter(buf, dtype=np.float64), q))
        t = w * local_t + (1.0 - w) * global_threshold
        thresh_used[i] = t
        weight_used[i] = w
        is_anom = mse_arr[i] > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse_arr[i])
    return preds, thresh_used, weight_used


def metrics(y_true, preds):
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) else float('nan')
    return {'precision': float(p), 'recall': float(r), 'f1': float(f1),
            'fpr': float(fpr), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)}

print('Threshold functions defined.')


## Step 4 — Calibrate PRIOR_STRENGTH on cc1_val (leak-free, target FPR≈1%)


In [ ]:
PRIOR_CANDIDATES = [100, 250, 500, 1000, 2000, 10000, 50000]

print('--- mean+k·std blend on cc1_val ---')
print(f'{"prior":>8s} {"FPR":>10s}')
prior_fpr_meank = {}
for ps in PRIOR_CANDIDATES:
    preds, _ = run_mean_k_blend(mse['cc1_val'], cmdb_ids['cc1_val'], prior_strength=ps)
    prior_fpr_meank[ps] = float(preds.mean())
    print(f'{ps:8d} {prior_fpr_meank[ps]*100:9.2f}%')
BEST_PRIOR_MEANK = min(PRIOR_CANDIDATES, key=lambda ps: abs(prior_fpr_meank[ps] - TARGET_FPR))
print(f'Chosen PRIOR_STRENGTH (mean+k) = {BEST_PRIOR_MEANK}')

print('\n--- FPR-targeted blend on cc1_val ---')
print(f'{"prior":>8s} {"FPR":>10s}')
prior_fpr_fprt = {}
for ps in PRIOR_CANDIDATES:
    preds, _, _ = run_fpr_targeted(mse['cc1_val'], cmdb_ids['cc1_val'], prior_strength=ps)
    prior_fpr_fprt[ps] = float(preds.mean())
    print(f'{ps:8d} {prior_fpr_fprt[ps]*100:9.2f}%')
BEST_PRIOR_FPRT = min(PRIOR_CANDIDATES, key=lambda ps: abs(prior_fpr_fprt[ps] - TARGET_FPR))
print(f'Chosen PRIOR_STRENGTH (FPR-targeted) = {BEST_PRIOR_FPRT}')


## Step 5 — Compare static vs mean+k·std vs FPR-targeted on cc1_test + drift_cc2


In [ ]:
results = {'static': {}, 'mean_k_blend': {}, 'fpr_targeted': {}}
print(f'{"set":12s} {"method":14s} {"P":>7s} {"R":>7s} {"F1":>7s} {"FPR":>8s} {"FP":>6s}')

for name in ALL_EVAL:
    static_pred = (mse[name] > STATIC_VAL_P99).astype(int)
    results['static'][name] = metrics(y[name], static_pred)
    m = results['static'][name]
    print(f'{name:12s} {"static":14s} {m["precision"]:7.3f} {m["recall"]:7.3f} {m["f1"]:7.3f} {m["fpr"]:8.4f} {m["fp"]:6d}')

    preds_mk, _ = run_mean_k_blend(mse[name], cmdb_ids[name], prior_strength=BEST_PRIOR_MEANK)
    results['mean_k_blend'][name] = metrics(y[name], preds_mk)
    m = results['mean_k_blend'][name]
    print(f'{name:12s} {"mean+k":14s} {m["precision"]:7.3f} {m["recall"]:7.3f} {m["f1"]:7.3f} {m["fpr"]:8.4f} {m["fp"]:6d}')

    preds_ft, thresh_ft, weight_ft = run_fpr_targeted(
        mse[name], cmdb_ids[name], prior_strength=BEST_PRIOR_FPRT)
    results['fpr_targeted'][name] = metrics(y[name], preds_ft)
    results['fpr_targeted'][name]['thresh_mean'] = float(thresh_ft.mean())
    results['fpr_targeted'][name]['weight_mean'] = float(weight_ft.mean())
    m = results['fpr_targeted'][name]
    print(f'{name:12s} {"fpr_target":14s} {m["precision"]:7.3f} {m["recall"]:7.3f} {m["f1"]:7.3f} {m["fpr"]:8.4f} {m["fp"]:6d}')
    print()

# Phase-1 ID gate
id_fprt = results['fpr_targeted']['cc1_test']
id_pr_auc = float(prior_eval['auc']['cc1_test']['auc_pr'])  # ranking unchanged by threshold
print('ID gate check (ranking PR-AUC unchanged by thresholding):')
print(f'  PR-AUC (from vae_eval) = {id_pr_auc:.4f}  (must stay >= 0.90)')
print(f'  FPR-targeted F1 = {id_fprt["f1"]:.3f}  (target >= 0.88)')
print(f'  Drift FPR-targeted FPR = {results["fpr_targeted"]["drift_cc2"]["fpr"]:.4f}  (target <= 0.03)')


## Step 6 — Per-fault recall (FPR-targeted vs static)


In [ ]:
per_fault = {}
for name in ALL_EVAL:
    per_fault[name] = {}
    preds_ft, _, _ = run_fpr_targeted(mse[name], cmdb_ids[name], prior_strength=BEST_PRIOR_FPRT)
    static_pred = (mse[name] > STATIC_VAL_P99).astype(int)
    for ftype in sorted({v for v in ft[name] if isinstance(v, str)}):
        mask = (ft[name] == ftype)
        n = int(mask.sum())
        per_fault[name][ftype] = {
            'n': n,
            'recall_static': float(static_pred[mask].mean()) if n else float('nan'),
            'recall_fpr_targeted': float(preds_ft[mask].mean()) if n else float('nan'),
        }
        print(f'{name:12s} {ftype:14s} n={n:4d}  static={per_fault[name][ftype]["recall_static"]:.3f}  '
              f'fpr_tgt={per_fault[name][ftype]["recall_fpr_targeted"]:.3f}')
    print()


## Step 7 — Save Phase-1 artifacts


In [ ]:
save_results = {
    'phase': 1,
    'method': 'fpr_targeted_blend',
    'window_size': WINDOW_SIZE,
    'buffer_size': BUFFER_SIZE,
    'target_fpr': TARGET_FPR,
    'min_local_for_percentile': MIN_LOCAL_FOR_PERCENTILE,
    'k_adaptive': K_ADAPTIVE,
    'static_val_p99': STATIC_VAL_P99,
    'prior_strength_mean_k': BEST_PRIOR_MEANK,
    'prior_strength_fpr_targeted': BEST_PRIOR_FPRT,
    'prior_fpr_search_mean_k': prior_fpr_meank,
    'prior_fpr_search_fpr_targeted': prior_fpr_fprt,
    'results': results,
    'per_fault_recall': per_fault,
    # Back-compat keys consumed by incremental_learning / final_comparison
    'prior_strength': BEST_PRIOR_FPRT,
    'threshold_mode': 'fpr_targeted',
}
# Keep legacy key layout for downstream notebooks that expect results[name].f1
save_results['results_legacy'] = {
    name: {k: results['fpr_targeted'][name][k] for k in ('precision', 'recall', 'f1', 'fpr', 'tp', 'fp', 'fn', 'tn')}
    for name in ALL_EVAL
}
# Also expose under results[name] for final_comparison compatibility (FPR-targeted as primary)
save_results['results'] = {
    name: {k: results['fpr_targeted'][name][k] for k in ('precision', 'recall', 'f1', 'fpr', 'tp', 'fp', 'fn', 'tn')}
    for name in ALL_EVAL
}
save_results['comparison_all_methods'] = results

out_path = os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

phase1_path = os.path.join(MODEL_DIR, 'phase1_fpr_targeted_eval.pkl')
with open(phase1_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {phase1_path}')


## Phase 1 summary

- **Primary method now deployed into `vae_cc1_adaptive_blended_eval.pkl`**: local FPR-targeted blend.
- Downstream Phase 2 (`incremental_learning.ipynb`) will read `prior_strength` + `threshold_mode` from this file.
- ID ranking (PR-AUC) is unchanged by any threshold; only operating-point metrics move.
